# Resume the Run, Not Just the Weights

Recover optimizer and random state, and import external tensors without silent transposes.

## Before You Start

Course 10. Entry course: 1. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** scratch-02_training.

## What You Will Be Able to Do

- Resume deterministic CPU updates
- Validate tokenizer and model metadata
- Map a foreign weight fixture explicitly

## Inference state is not training state

Weights alone are sufficient for a deterministic forward pass given the architecture, preprocessing, and mode. Continuing training also depends on optimizer moments, step number, schedule, random-number generators, and data position. A shuffled sampler and dropout consume randomness. Mixed-precision runs can additionally require a gradient-scaler state. The experiment restores a CPU Torch generator and Adam state; distributed determinism is a separate challenge.

A useful checkpoint records model config, tokenizer fingerprint, model tensors, optimizer state, completed updates, RNG state, and data cursor. Validate identity before applying arrays. Never load untrusted pickle-based files with unrestricted deserialization. Our browser lab uses ordinary arrays and dictionaries; it is about compatibility checks, not a secure serialization format.

## External weights need a declared mapping

A linear map may be stored as [out,in] in one library and [in,out] in another. Similar parameter names are not proof of equal meaning. Explicitly map source names to destination names, specify transpose operations, validate shapes and finite values, and reject missing or unexpected tensors. Square matrices can hide orientation mistakes, so tests must use rectangular fixtures and numerical forward equivalence.

Our import_weights function accepts a source dictionary, a destination specification of shapes, and a mapping destination -> (source_name, transpose). It returns independent arrays. Every source key and every destination key must appear exactly once. This deliberately small contract excludes split/fused QKV, tensor parallel shards, and quantized formats. Real GPT-2 downloads remain optional: the mechanism is learnable without network access or model-license assumptions.

## Worked Example: A rectangular projection

A source stores W=[[1,2],[3,4],[5,6]] with shape [3,2]. The destination expects [2,3].

1. **Declare orientation:** The map explicitly requests transpose. Destination W becomes [[1,3,5],[2,4,6]].
2. **Check computation:** For x=[10,20], x @ destination_W equals [50,110,170], the same as source_W @ x.
3. **Test continuation:** Save after step k, run step k+1, restore into fresh objects, repeat the same sampled batch, and compare parameters and loss. A fresh Adam optimizer should fail this equivalence.

A load that succeeds syntactically can still mean a different model or a different training run.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Explicit local weight-import contract; no deserialization of external files."""
import numpy as np

def import_weights(source, shapes, mapping):
    """Return copied destination arrays from an explicit tensor mapping.

source: {name: finite ndarray}; shapes: {destination: expected shape tuple}.
mapping: {destination: (source_name, transpose_bool)}. A transpose requires
a 2D array. Every source and destination must occur exactly once.
Reject missing/extra keys, repeated sources, invalid flags, nonfinite values,
or wrong shapes with ValueError. Do not mutate/alias source arrays."""
    raise NotImplementedError('Implement this function, then run the checks.')

def validate_checkpoint(checkpoint, config, tokenizer_id):
    """Validate a metadata envelope and return completed update count.

Require keys model, optimizer, rng, cursor, step, config, tokenizer_id.
config and tokenizer_id must exactly equal expected values. step/cursor
are nonnegative integers (not bool). Missing keys/mismatch: ValueError.
This checks metadata only, not tensor integrity or deserialization safety."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

A dense projection expects [input=4, output=7].

- Number of weight values (no bias)

### Worked Answers

- 28: 4*7=28. A matching count alone does not prove matching orientation.

## Experiment

Run `experiments/scratch_training.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- The training experiment performs one uninterrupted and one restored continuation and asserts matching CPU losses and tensors.
- Change only the saved optimizer state and observe the discrepancy. Exact reproducibility across devices and library releases is not promised.

## Transfer Problems

1. Add a data-order fingerprint and reject a resume against a different corpus.
2. Extend the mapping to a fused QKV fixture with explicit split offsets and independent forward checks.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[PyTorch: general checkpoints](https://pytorch.org/tutorials/recipes/recipes/saving_and_loading_a_general_checkpoint.html). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.


## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Explicit local weight-import contract; no deserialization of external files."""
import numpy as np

def import_weights(source, shapes, mapping):
    """Return copied destination arrays from an explicit tensor mapping.

source: {name: finite ndarray}; shapes: {destination: expected shape tuple}.
mapping: {destination: (source_name, transpose_bool)}. A transpose requires
a 2D array. Every source and destination must occur exactly once.
Reject missing/extra keys, repeated sources, invalid flags, nonfinite values,
or wrong shapes with ValueError. Do not mutate/alias source arrays."""
    raise NotImplementedError('Implement this function, then run the checks.')

def validate_checkpoint(checkpoint, config, tokenizer_id):
    """Validate a metadata envelope and return completed update count.

Require keys model, optimizer, rng, cursor, step, config, tokenizer_id.
config and tokenizer_id must exactly equal expected values. step/cursor
are nonnegative integers (not bool). Missing keys/mismatch: ValueError.
This checks metadata only, not tensor integrity or deserialization safety."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np
src = {'foreign.weight': np.arange(6.).reshape(3, 2), 'foreign.bias': np.array([1., 2., 3.])}
shapes = {'weight': (2, 3), 'bias': (3,)}
mapping = {'weight': ('foreign.weight', True), 'bias': ('foreign.bias', False)}
got = import_weights(src, shapes, mapping)
assert np.array_equal(np.array([10., 20.]) @ got['weight'], src['foreign.weight'] @ np.array([10., 20.]))
got['weight'][0, 0] = -99
assert src['foreign.weight'][0, 0] == 0, 'Imported tensors must not alias sources'
ck = dict(model={}, optimizer={}, rng=[1], cursor=3, step=7, config={'width': 8}, tokenizer_id='bytes-v1')
assert validate_checkpoint(ck, {'width': 8}, 'bytes-v1') == 7
bad_cases = [lambda: import_weights(src, shapes, {**mapping, 'weight': ('foreign.weight', False)}),
             lambda: import_weights({**src, 'extra': np.zeros(1)}, shapes, mapping),
             lambda: import_weights(src, shapes, {'weight': ('foreign.weight', True), 'bias': ('foreign.weight', False)}),
             lambda: validate_checkpoint(ck, {'width': 16}, 'bytes-v1'),
             lambda: validate_checkpoint(ck, {'width': 8}, 'same-size-different-vocab'),
             lambda: validate_checkpoint({k: v for k, v in ck.items() if k != 'optimizer'}, {'width': 8}, 'bytes-v1')]
for call in bad_cases:
    try:
        call()
    except ValueError:
        pass
    else:
        raise AssertionError('Reject incompatible artifacts before loading')
print('Checkpoint and weight-import checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Explicit local weight-import contract; no deserialization of external files."""
import numpy as np


def import_weights(source, shapes, mapping):
    """Return copied destination arrays from an explicit tensor mapping.

    source: {name: finite ndarray}; shapes: {destination: expected shape tuple}.
    mapping: {destination: (source_name, transpose_bool)}. A transpose requires
    a 2D array. Every source and destination must occur exactly once.
    Reject missing/extra keys, repeated sources, invalid flags, nonfinite values,
    or wrong shapes with ValueError. Do not mutate/alias source arrays.
    """
    if set(mapping) != set(shapes):
        raise ValueError('destination mapping mismatch')
    names = [entry[0] for entry in mapping.values()]
    if set(names) != set(source) or len(names) != len(set(names)):
        raise ValueError('source mapping must be bijective')
    result = {}
    for destination, (name, transpose) in mapping.items():
        value = np.asarray(source[name])
        if not isinstance(transpose, bool) or (transpose and value.ndim != 2):
            raise ValueError('transpose requires a matrix and Boolean flag')
        value = value.T if transpose else value
        if value.shape != tuple(shapes[destination]) or not np.isfinite(value).all():
            raise ValueError('incompatible tensor')
        result[destination] = value.copy()
    return result


def validate_checkpoint(checkpoint, config, tokenizer_id):
    """Validate a metadata envelope and return completed update count.

    Require keys model, optimizer, rng, cursor, step, config, tokenizer_id.
    config and tokenizer_id must exactly equal expected values. step/cursor
    are nonnegative integers (not bool). Missing keys/mismatch: ValueError.
    This checks metadata only, not tensor integrity or deserialization safety.
    """
    required = {'model', 'optimizer', 'rng', 'cursor', 'step', 'config', 'tokenizer_id'}
    if not required <= checkpoint.keys() or checkpoint['config'] != config or checkpoint['tokenizer_id'] != tokenizer_id:
        raise ValueError('incompatible checkpoint metadata')
    for key in ('step', 'cursor'):
        value = checkpoint[key]
        if not isinstance(value, int) or isinstance(value, bool) or value < 0:
            raise ValueError('invalid progress state')
    return checkpoint['step']


In [ ]:
import numpy as np
src = {'foreign.weight': np.arange(6.).reshape(3, 2), 'foreign.bias': np.array([1., 2., 3.])}
shapes = {'weight': (2, 3), 'bias': (3,)}
mapping = {'weight': ('foreign.weight', True), 'bias': ('foreign.bias', False)}
got = import_weights(src, shapes, mapping)
assert np.array_equal(np.array([10., 20.]) @ got['weight'], src['foreign.weight'] @ np.array([10., 20.]))
got['weight'][0, 0] = -99
assert src['foreign.weight'][0, 0] == 0, 'Imported tensors must not alias sources'
ck = dict(model={}, optimizer={}, rng=[1], cursor=3, step=7, config={'width': 8}, tokenizer_id='bytes-v1')
assert validate_checkpoint(ck, {'width': 8}, 'bytes-v1') == 7
bad_cases = [lambda: import_weights(src, shapes, {**mapping, 'weight': ('foreign.weight', False)}),
             lambda: import_weights({**src, 'extra': np.zeros(1)}, shapes, mapping),
             lambda: import_weights(src, shapes, {'weight': ('foreign.weight', True), 'bias': ('foreign.weight', False)}),
             lambda: validate_checkpoint(ck, {'width': 16}, 'bytes-v1'),
             lambda: validate_checkpoint(ck, {'width': 8}, 'same-size-different-vocab'),
             lambda: validate_checkpoint({k: v for k, v in ck.items() if k != 'optimizer'}, {'width': 8}, 'bytes-v1')]
for call in bad_cases:
    try:
        call()
    except ValueError:
        pass
    else:
        raise AssertionError('Reject incompatible artifacts before loading')
print('Checkpoint and weight-import checks passed.')


## Existing Course Model

The exact vendored mini_llm configuration and model are embedded here so the notebook does not depend on a repository clone or network import. The Torch model includes learned normalization affine parameters and exact GELU; the browser capstone deliberately omits affine parameters and uses its documented approximation.


In [ ]:
from dataclasses import asdict, dataclass
from typing import Dict, Any


@dataclass
class ModelConfig:
    vocab_size: int
    block_size: int = 64
    n_layer: int = 2
    n_head: int = 4
    n_embd: int = 64
    dropout: float = 0.1
    bias: bool = False

    def __post_init__(self) -> None:
        if self.n_embd % self.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        if self.vocab_size <= 0 or self.block_size <= 0:
            raise ValueError("vocab_size and block_size must be positive")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

"""A bigram baseline and a small decoder-only Transformer."""
import math
from typing import Dict, Optional, Tuple
import torch
import torch.nn as nn
from torch.nn import functional as F

def sample_next_token(logits: torch.Tensor, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
    if temperature <= 0:
        raise ValueError('temperature must be greater than zero')
    logits = logits / temperature
    if top_k is not None:
        top_k = min(top_k, logits.size(-1))
        threshold = torch.topk(logits, top_k).values[:, [-1]]
        logits = logits.masked_fill(logits < threshold, float('-inf'))
    probabilities = F.softmax(logits, dim=-1)
    return torch.multinomial(probabilities, num_samples=1)

class BigramLanguageModel(nn.Module):
    """Each token directly stores logits for the token that follows it."""

    def __init__(self, vocab_size: int) -> None:
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, vocab_size)
        nn.init.zeros_(self.token_embedding.weight)
        self.vocab_size = vocab_size

    def forward(self, inputs: torch.Tensor, targets: Optional[torch.Tensor]=None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        logits = self.token_embedding(inputs)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), targets.reshape(-1))
        return (logits, loss)

    @torch.no_grad()
    def generate(self, tokens: torch.Tensor, max_new_tokens: int, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
        for _ in range(max_new_tokens):
            logits, _ = self(tokens[:, -1:])
            next_token = sample_next_token(logits[:, -1, :], temperature, top_k)
            tokens = torch.cat((tokens, next_token), dim=1)
        return tokens

class CausalSelfAttention(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.query_key_value = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.projection = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attention_dropout = nn.Dropout(config.dropout)
        self.residual_dropout = nn.Dropout(config.dropout)
        causal_mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer('causal_mask', causal_mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        batch_size, time_steps, channels = inputs.shape
        qkv = self.query_key_value(inputs)
        query, key, value = qkv.split(self.n_embd, dim=2)

        def split_heads(tensor: torch.Tensor) -> torch.Tensor:
            return tensor.view(batch_size, time_steps, self.n_head, self.head_dim).transpose(1, 2)
        query, key, value = map(split_heads, (query, key, value))
        attention = query @ key.transpose(-2, -1) / math.sqrt(self.head_dim)
        attention = attention.masked_fill(self.causal_mask[:, :, :time_steps, :time_steps] == 0, float('-inf'))
        weights = self.attention_dropout(F.softmax(attention, dim=-1))
        output = weights @ value
        output = output.transpose(1, 2).contiguous().view(batch_size, time_steps, channels)
        return self.residual_dropout(self.projection(output))

class FeedForward(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias), nn.GELU(), nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias), nn.Dropout(config.dropout))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.layers(inputs)

class TransformerBlock(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.attention_norm = nn.LayerNorm(config.n_embd)
        self.attention = CausalSelfAttention(config)
        self.feed_forward_norm = nn.LayerNorm(config.n_embd)
        self.feed_forward = FeedForward(config)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        inputs = inputs + self.attention(self.attention_norm(inputs))
        inputs = inputs + self.feed_forward(self.feed_forward_norm(inputs))
        return inputs

class GPT(nn.Module):
    """A compact GPT-2-style decoder-only Transformer."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        self.final_norm = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._initialize_weights)

    @staticmethod
    def _initialize_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, inputs: torch.Tensor, targets: Optional[torch.Tensor]=None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        _, time_steps = inputs.shape
        if time_steps > self.config.block_size:
            raise ValueError(f'Sequence length {time_steps} exceeds block size {self.config.block_size}')
        positions = torch.arange(time_steps, device=inputs.device)
        hidden = self.token_embedding(inputs) + self.position_embedding(positions)
        hidden = self.dropout(hidden)
        for block in self.blocks:
            hidden = block(hidden)
        logits = self.lm_head(self.final_norm(hidden))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.config.vocab_size), targets.reshape(-1))
        return (logits, loss)

    def parameter_count(self, exclude_position_embeddings: bool=True) -> int:
        count = sum((parameter.numel() for parameter in self.parameters()))
        if exclude_position_embeddings:
            count -= self.position_embedding.weight.numel()
        return count

    def optimizer_groups(self, weight_decay: float) -> list:
        decay, no_decay = ([], [])
        for parameter in self.parameters():
            (decay if parameter.dim() >= 2 else no_decay).append(parameter)
        return [{'params': decay, 'weight_decay': weight_decay}, {'params': no_decay, 'weight_decay': 0.0}]

    @torch.no_grad()
    def generate(self, tokens: torch.Tensor, max_new_tokens: int, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
        for _ in range(max_new_tokens):
            context = tokens[:, -self.config.block_size:]
            logits, _ = self(context)
            next_token = sample_next_token(logits[:, -1, :], temperature, top_k)
            tokens = torch.cat((tokens, next_token), dim=1)
        return tokens

def build_model(model_type: str, config: ModelConfig) -> nn.Module:
    if model_type == 'bigram':
        return BigramLanguageModel(config.vocab_size)
    if model_type == 'gpt':
        return GPT(config)
    raise ValueError("model_type must be 'bigram' or 'gpt'")

def checkpoint_model(checkpoint: Dict[str, object], device: torch.device) -> nn.Module:
    raw_config = checkpoint.get('model_config')
    model_type = checkpoint.get('model_type')
    if not isinstance(raw_config, dict) or not isinstance(model_type, str):
        raise ValueError('Checkpoint is missing model metadata')
    model = build_model(model_type, ModelConfig(**raw_config))
    model.load_state_dict(checkpoint['model_state'])
    return model.to(device)


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""CPU pretraining, token-weighted accumulation, and exact local resume.

Run: uv run --with numpy --with torch python experiments/scratch_training.py
The notebook embeds the existing mini_llm model source before this cell.
No downloaded data/weights. Fixed synthetic language, not natural-language skill.
"""
import copy
import hashlib
import io
import json
from pathlib import Path
import sys
import torch
from torch.nn import functional as F

if 'GPT' not in globals():
    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'colab/vendor/src'))
    from mini_llm.models import GPT
    from mini_llm.config import ModelConfig


def scratch_data(seed=19):
    """Independent unique sequences: next symbol usually advances by one."""
    generator = torch.Generator().manual_seed(seed)
    rows, seen = [], set()
    while len(rows) < 320:
        row = [int(torch.randint(8, (1,), generator=generator))]
        for _ in range(12):
            step = 1 if torch.rand((), generator=generator) < .82 else 3
            row.append((row[-1] + step) % 8)
        if tuple(row) not in seen:
            seen.add(tuple(row)); rows.append(row)
    data = torch.tensor(rows)
    permutation = torch.randperm(len(data), generator=generator)
    data = data[permutation]
    return data[:224], data[224:272], data[272:]


def scratch_pretrain(seed=7, updates=140):
    torch.set_num_threads(1)
    torch.manual_seed(seed)
    config = ModelConfig(vocab_size=8, block_size=12, n_layer=1, n_head=2, n_embd=24, dropout=0.)
    model = GPT(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=.006, weight_decay=.01)
    train, validation, test = scratch_data()
    generator = torch.Generator().manual_seed(seed + 1)

    def evaluate(rows):
        model.eval()
        with torch.no_grad():
            logits, _ = model(rows[:, :-1])
            return float(F.cross_entropy(logits.flatten(0, 1), rows[:, 1:].flatten()))

    def next_update(m, opt, gen):
        m.train()
        indices = torch.randint(len(train), (24,), generator=gen)
        batch = train[indices]
        opt.zero_grad(set_to_none=True)
        # Unequal microbatch sizes still implement the full token-average gradient.
        tokens = batch[:, 1:].numel()
        total_loss = 0.
        for micro in (batch[:7], batch[7:]):
            logits, _ = m(micro[:, :-1])
            loss_sum = F.cross_entropy(logits.flatten(0, 1), micro[:, 1:].flatten(), reduction='sum')
            loss = loss_sum / tokens
            loss.backward()
            total_loss += float(loss.detach())
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.)
        opt.step()
        return total_loss

    initial = evaluate(validation)
    history = []
    for step in range(updates):
        next_update(model, optimizer, generator)
        if (step + 1) % 35 == 0:
            history.append({'update': step + 1, 'train_nll': evaluate(train), 'validation_nll': evaluate(validation)})

    # Snapshot immediately before the next sampled update, including RNG state.
    checkpoint = copy.deepcopy(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
        rng=generator.get_state(), torch_rng=torch.get_rng_state(), step=updates, cursor=0,
        config=config.to_dict(), tokenizer_id='symbols-a-through-h-v1'))
    serialized = io.BytesIO()
    torch.save(checkpoint, serialized)
    serialized.seek(0)
    checkpoint = torch.load(serialized, weights_only=True)

    continued = copy.deepcopy(model)
    continued_optimizer = torch.optim.AdamW(continued.parameters(), lr=.006, weight_decay=.01)
    continued_optimizer.load_state_dict(copy.deepcopy(optimizer.state_dict()))
    continued_rng = torch.Generator(); continued_rng.set_state(generator.get_state())
    with torch.random.fork_rng(devices=[]):
        restored = GPT(ModelConfig(**checkpoint['config']))
        restored.load_state_dict(checkpoint['model'])
        resumed_optimizer = torch.optim.AdamW(restored.parameters(), lr=.006, weight_decay=.01)
        resumed_optimizer.load_state_dict(checkpoint['optimizer'])
        resumed_rng = torch.Generator(); resumed_rng.set_state(checkpoint['rng'])
        torch.set_rng_state(checkpoint['torch_rng'])
        uninterrupted = next_update(continued, continued_optimizer, continued_rng)
        torch.set_rng_state(checkpoint['torch_rng'])
        resumed = next_update(restored, resumed_optimizer, resumed_rng)
    assert uninterrupted == resumed
    assert all(torch.equal(a, b) for a, b in zip(continued.parameters(), restored.parameters()))

    # Independent gradient-equivalence check before clipping/optimizer state.
    model.eval()
    batch = train[:9]
    optimizer.zero_grad(set_to_none=True)
    _, full_loss = model(batch[:, :-1], batch[:, 1:]); full_loss.backward()
    full_grads = [p.grad.clone() for p in model.parameters()]
    optimizer.zero_grad(set_to_none=True)
    for micro in (batch[:2], batch[2:]):
        logits, _ = model(micro[:, :-1])
        (F.cross_entropy(logits.flatten(0, 1), micro[:, 1:].flatten(), reduction='sum') / batch[:, 1:].numel()).backward()
    error = max(float((p.grad - g).abs().max()) for p, g in zip(model.parameters(), full_grads))
    assert error < 2e-6, error
    final = evaluate(validation)
    assert final < initial, 'This fixed toy run should learn its transition pattern'
    model.eval(); torch.manual_seed(seed + 100)
    sample = model.generate(torch.tensor([[0]]), 16, top_k=3).tolist()[0]
    report = dict(seed=seed, updates=updates, config=config.to_dict(), parameters=model.parameter_count(False),
        split_sizes=[len(train), len(validation), len(test)],
        data_sha256=hashlib.sha256(torch.cat([train, validation, test]).numpy().tobytes()).hexdigest(),
        initial_validation_nll=initial, final_validation_nll=final, test_nll=evaluate(test),
        accumulation_max_error=error, resume_exact=True, history=history, generated_ids=sample,
        tokenizer=dict(symbols=list('abcdefgh'),ids=list(range(8))),generated_text=''.join('abcdefgh'[i] for i in sample),
        limitation='Synthetic transition language; not a general-purpose LLM. Test used once after a fixed recipe.')
    return model, report


if __name__ == '__main__':
    _, report = scratch_pretrain()
    print(json.dumps(report, indent=2))
